# Chapter 10 - Qwen Omni Fine-Tuning Notebook

We isolated the training of Qwen Omni in a single Notebook, the reason behind it is that it relies on particular versions of some libraries.

In [1]:
# We have to fix some versions of the libraries to make the training script work.
# Important: remember to restart your session after running this cell!
!pip install --upgrade pip

!pip install \
  "torch==2.6.0" \
  "torchvision==0.21.0" \
  "torchaudio==2.6.0" \
  --index-url https://download.pytorch.org/whl/cu124

!pip install \
  "transformers==4.57.1" \
  "datasets==3.6.0" \
  "peft==0.18.0" \
  "trl==0.25.1" \
  "accelerate==1.11.0" \
  "qwen-omni-utils==0.0.8" \
  "librosa==0.10.2" \
  "soundfile==0.12.1" \
  "numpy==2.2.6"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 90.8 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
Looking in indexes: https://download.pytorch.org/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 214.8 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 39.2 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 223.4 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 37.5 MB/s  0:00:07
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 136.6 MB/s  0:00:02
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 167.5 MB/s  0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 182.4 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 89.8 MB/s  0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 162.2 MB/s  0:00:01
 

## Imports and configuration

In the cell below we set some basic configuration such as the base model we are going to use and the dataset that we will use for training

In [1]:
import numpy as np
import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from transformers import AutoProcessor, Qwen2_5OmniThinkerForConditionalGeneration, Trainer, TrainingArguments

# Configuration
MODEL_ID = "Qwen/Qwen2.5-Omni-3B"
DERIVED_DATASET_ID = "orrzohar/EMID-Emotion-Matching"
OUTPUT_DIR = "./book_demo_output"
MAX_EVAL_EXAMPLES = 200
SEED = 42

SYSTEM_PROMPT = (
    "You are Qwen, a virtual human developed by the Qwen Team, Alibaba Group, capable of perceiving auditory and "
    "visual inputs, as well as generating text and speech."
)

## Helper functions
Below some functionalities to prepare audio samples and the collator function. Pay special attention to that one!

In [2]:
import numpy as np
import torch
import torchaudio

def _mixdown_to_mono(x):
    if isinstance(x, torch.Tensor):
        if x.ndim == 1:
            return x
        if x.ndim == 2:
            # Usually [channels, time] for torchcodec/torchaudio
            if x.shape[0] <= 8 and x.shape[1] > x.shape[0]:
                return x.mean(dim=0)
            return x.mean(dim=-1)
        return x.reshape(-1)

    x = np.asarray(x, dtype=np.float32)
    if x.ndim == 1:
        return x
    if x.ndim == 2:
        if x.shape[0] <= 8 and x.shape[1] > x.shape[0]:
            return x.mean(axis=0)
        return x.mean(axis=-1)
    return x.reshape(-1)

def to_waveform(audio, processor) -> np.ndarray:
    """
    Convert HF audio payloads / AudioDecoder / tensors / arrays into a mono
    float32 waveform at the processor's expected sampling rate.
    """
    target_sr = int(processor.feature_extractor.sampling_rate)
    wave = None
    sample_rate = None

    # HF-style decoded dict: {"array": ..., "sampling_rate": ...}
    if isinstance(audio, dict):
        if audio.get("array") is not None:
            wave = audio["array"]
            sample_rate = audio.get("sampling_rate")
        elif audio.get("path"):
            wave, sample_rate = torchaudio.load(audio["path"])
        else:
            raise ValueError(f"Unsupported audio dict keys: {list(audio.keys())}")

    # TorchCodec / HF AudioDecoder object
    elif hasattr(audio, "get_all_samples"):
        audio_samples = audio.get_all_samples()
        wave = audio_samples.data
        sample_rate = getattr(audio_samples, "sample_rate", None)
        if sample_rate is None and hasattr(audio, "metadata"):
            sample_rate = getattr(audio.metadata, "sample_rate", None)

    # Some wrappers expose .array / .sampling_rate
    elif hasattr(audio, "array"):
        wave = audio.array
        sample_rate = getattr(audio, "sampling_rate", None)

    # Some wrappers expose a .path
    elif hasattr(audio, "path") and getattr(audio, "path"):
        wave, sample_rate = torchaudio.load(audio.path)

    elif isinstance(audio, torch.Tensor):
        wave = audio

    elif isinstance(audio, (np.ndarray, list, tuple)):
        wave = audio

    else:
        raise TypeError(f"Unsupported audio type: {type(audio)}")

    wave = _mixdown_to_mono(wave)

    if isinstance(wave, torch.Tensor):
        wave = wave.detach().cpu().float()
        if sample_rate is not None and int(sample_rate) != target_sr:
            wave = torchaudio.functional.resample(
                wave.unsqueeze(0), int(sample_rate), target_sr
            ).squeeze(0)
        return wave.numpy().astype(np.float32, copy=False)

    wave = np.asarray(wave, dtype=np.float32)
    if sample_rate is not None and int(sample_rate) != target_sr:
        wave_t = torch.from_numpy(wave).float().unsqueeze(0)
        wave = torchaudio.functional.resample(
            wave_t, int(sample_rate), target_sr
        ).squeeze(0).numpy()
    return wave.astype(np.float32, copy=False)

def format_messages(question: str, same: bool, emotion: str, train: bool):
    base = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user", "content": [{"type": "audio"}, {"type": "image"}, {"type": "text", "text": question}]},
    ]
    if not train:
        return base
    target = f"yes - {emotion}" if same else "no"
    return base + [{"role": "assistant", "content": [{"type": "text", "text": target}]}]

def build_collator(processor):
    tok = processor.tokenizer
    pad_id = tok.pad_token_id
    im_start = tok.convert_tokens_to_ids("<|im_start|>")
    im_end = tok.convert_tokens_to_ids("<|im_end|>")

    def to_ids(role):
        enc = tok(f"{role}\n", add_special_tokens=False)
        ids = enc.input_ids if hasattr(enc, "input_ids") else enc["input_ids"]
        return tuple(ids[0] if isinstance(ids[0], list) else ids)

    role_ids = {role: to_ids(role) for role in ("system", "user", "assistant")}

    def mask(ids):
        labels = ids.clone()
        role = None
        i = 0
        while i < labels.size(0):
            token = ids[i].item()
            if token == im_start:
                labels[i] = -100
                i += 1
                for name, seq in role_ids.items():
                    if ids[i : i + len(seq)].tolist() == list(seq):
                        labels[i : i + len(seq)] = -100
                        role = name
                        i += len(seq)
                        break
                continue
            labels[i] = ids[i] if role == "assistant" else -100
            if token == im_end:
                role = None
            i += 1
        if pad_id is not None:
            labels[ids == pad_id] = -100
        return labels

    def collate(batch):
        texts = [
            processor.apply_chat_template(
                format_messages(item["question"], bool(item["same"]), item["emotion"], True),
                tokenize=False,
                add_generation_prompt=False,
            )
            for item in batch
        ]
        encoded = processor(
            text=texts,
            audio=[to_waveform(item["audio"], processor) for item in batch],
            images=[item["image"] for item in batch],
            padding=True,
            return_tensors="pt",
        )
        encoded["labels"] = torch.stack([mask(ids) for ids in encoded["input_ids"]])
        return encoded

    return collate


## Evaluation helpers
Below you can find some functions that we use to clean the output of the model and evaluate it against the particular dataset that we chose as example in this notebook

In [3]:
import re

def normalize_emotion(s: str) -> str:
    s = str(s).strip().lower()

    for marker in (
        "<|im_end|>",
        "<|endoftext|>",
        "human:",
        "user:",
        "assistant:",
        "system:",
    ):
        if marker in s:
            s = s.split(marker, 1)[0]

    s = s.splitlines()[0].strip() if s else ""

    for stop in (".", ",", ";", "!", "?"):
        if stop in s:
            s = s.split(stop, 1)[0]

    s = s.strip(" \t\r\n-–—:_'\"`()[]{}")
    s = (
        s.replace("-", "_")
         .replace("–", "_")
         .replace("—", "_")
         .replace("/", "_")
         .replace(" ", "_")
    )

    while "__" in s:
        s = s.replace("__", "_")

    return s.strip("_")


def parse_completion(completion: str):
    c = str(completion).strip().lower()

    for marker in (
        "<|im_end|>",
        "<|endoftext|>",
        "human:",
        "user:",
        "assistant:",
        "system:",
    ):
        if marker in c:
            c = c.split(marker, 1)[0]

    c = c.strip()
    first_line = c.splitlines()[0].strip() if c else ""

    if first_line.startswith("yes"):
        tail = first_line[len("yes"):].lstrip(" \t-–—:,.")
        pred_emotion = normalize_emotion(tail) if tail else ""
        return True, pred_emotion

    if first_line.startswith("no"):
        return False, ""

    return False, ""


def build_gen_kwargs(processor, max_new_tokens: int = 12):
    tok = processor.tokenizer

    pad_token_id = tok.pad_token_id
    if pad_token_id is None:
        pad_token_id = tok.eos_token_id

    im_end_id = tok.convert_tokens_to_ids("<|im_end|>")

    eos_ids = []
    for tid in (tok.eos_token_id, im_end_id):
        if isinstance(tid, int) and tid >= 0 and tid not in eos_ids:
            eos_ids.append(tid)

    gen_kwargs = {
        "max_new_tokens": max_new_tokens,
        "do_sample": False,
        "pad_token_id": pad_token_id,
    }

    if len(eos_ids) == 1:
        gen_kwargs["eos_token_id"] = eos_ids[0]
    elif len(eos_ids) > 1:
        gen_kwargs["eos_token_id"] = eos_ids

    return gen_kwargs


def evaluate(model, processor, records):
    device = next(model.parameters()).device
    gen_kwargs = build_gen_kwargs(processor, max_new_tokens=12)

    total = min(len(records), MAX_EVAL_EXAMPLES)
    same_hits = 0
    emotion_hits = 0
    joint_hits = 0
    positives = 0

    was_training = model.training
    model.eval()

    try:
        with torch.no_grad():
            for idx in range(total):
                sample = records[idx]
                is_same = bool(sample["same"])
                actual_emotion = normalize_emotion(sample["emotion"])

                if is_same:
                    positives += 1

                prompt = processor.apply_chat_template(
                    format_messages(
                        sample["question"],
                        is_same,
                        sample["emotion"],
                        train=False,
                    ),
                    tokenize=False,
                    add_generation_prompt=True,
                )

                inputs = processor(
                    text=[prompt],
                    audio=[to_waveform(sample["audio"], processor)],
                    images=[sample["image"]],
                    return_tensors="pt",
                ).to(device)

                output = model.generate(**inputs, **gen_kwargs)

                completion = processor.batch_decode(
                    output[:, inputs["input_ids"].shape[-1]:],
                    skip_special_tokens=False,
                    clean_up_tokenization_spaces=False,
                )[0]

                pred_same, pred_emotion = parse_completion(completion)

                if pred_same == is_same:
                    same_hits += 1

                    if not is_same:
                        joint_hits += 1
                    elif pred_emotion == actual_emotion:
                        emotion_hits += 1
                        joint_hits += 1
    finally:
        if was_training:
            model.train()

    return (
        same_hits / total if total else 0.0,
        emotion_hits / positives if positives else 0.0,
        joint_hits / total if total else 0.0,
    )


## Prepare model and processor

In [4]:


processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
thinker = Qwen2_5OmniThinkerForConditionalGeneration.from_pretrained(MODEL_ID, trust_remote_code=True, torch_dtype=dtype)
thinker.to(dtype=dtype, device=torch.device("cuda" if torch.cuda.is_available() else "cpu"))

datasets_dict = load_dataset(DERIVED_DATASET_ID)
train_dataset = datasets_dict["train"]
eval_dataset = datasets_dict["test"]


preprocessor_config.json:   0%|          | 0.00/667 [00:00<?, ?B/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

data/test-00004-of-00006.parquet:   0%|          | 0.00/483M [00:00<?, ?B/s]

data/test-00005-of-00006.parquet:   0%|          | 0.00/481M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/24000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6000 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/22 [00:00<?, ?it/s]

## We evaluate the base model in the evaluation fragment of the dataset
You will use that later to compare performance against the fine-tuned model

In [5]:
baseline = evaluate(thinker, processor, eval_dataset)

## Train code

In [6]:
import inspect
import os
import shutil

# Make reruns deterministic and avoid output-dir conflicts on older transformers builds
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)

lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)

thinker = get_peft_model(thinker, lora_cfg)

supported = inspect.signature(TrainingArguments.__init__).parameters

ta_kwargs = {
    "output_dir": OUTPUT_DIR,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 4,
    "num_train_epochs": 1,
    "learning_rate": 1e-5,
    "logging_steps": 2,
    "remove_unused_columns": False,
}

optional_kwargs = {
    "save_total_limit": 1,
    "data_seed": SEED,
    "report_to": "none",
    "overwrite_output_dir": True,
    "bf16": bool(torch.cuda.is_available() and getattr(torch.cuda, "is_bf16_supported", lambda: False)()),
    "fp16": bool(torch.cuda.is_available() and not getattr(torch.cuda, "is_bf16_supported", lambda: False)()),
}

for k, v in optional_kwargs.items():
    if k in supported:
        ta_kwargs[k] = v

trainer = Trainer(
    model=thinker,
    args=TrainingArguments(**ta_kwargs),
    train_dataset=train_dataset,
    data_collator=build_collator(processor),
)

trainer.train()
tuned = evaluate(trainer.model, processor, eval_dataset)

print("\n=== EMID Music ↔ Image Emotion Alignment ===")
print(f"{'Model':<12} | {'Same Acc':>8} | {'Joint Acc':>9} | {'Emotion Acc':>11}")
print("-" * 50)
for name, scores in (("Base", baseline), ("LoRA", tuned)):
    print(f"{name:<12} | {scores[0]*100:8.1f}% | {scores[2]*100:9.1f}% | {scores[1]*100:11.1f}%")
print(
    f"\nΔ same accuracy: {(tuned[0] - baseline[0])*100:+.1f}% | "
    f"Δ joint accuracy: {(tuned[2] - baseline[2])*100:+.1f}% | "
    f"Δ emotion accuracy: {(tuned[1] - baseline[1])*100:+.1f}%"
)

Step,Training Loss
2,11.120700
4,12.752000
6,12.770500
8,9.409800
10,8.352200
12,7.326300
14,5.519600
16,4.076400
18,3.979900
20,3.921200


Step,Training Loss
2,11.120700
4,12.752000
6,12.770500
8,9.409800
10,8.352200
12,7.326300
14,5.519600
16,4.076400
18,3.979900
20,3.921200



=== EMID Music ↔ Image Emotion Alignment ===
Model        | Same Acc | Joint Acc | Emotion Acc
--------------------------------------------------
Base         |     49.0% |      46.5% |         0.0%
LoRA         |     79.0% |      76.5% |        75.5%

Δ same accuracy: +30.0% | Δ joint accuracy: +30.0% | Δ emotion accuracy: +75.5%
